# Step 1: Get Features From Multiple Datasets
- Using [pybiber](https://pypi.org/project/pybiber/)

In [1]:
# HuggingFace Login
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
hf_token = os.getenv("HUGGING_FACE_TOKEN")
login(hf_token)

In [16]:
import pybiber as pb
import polars as pl
import os
import numpy as np
import random
import torch
from scipy.stats import zscore
import pandas as pd
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import gc

In [3]:
DEVICE = 0 if torch.cuda.is_available() else -1
FILE_PATH = 'getText/datasetsPrep'
OUTPUT_DIR = 'biberOutputs'
SAMPLE_SIZE = 10
batch_idx = 0

In [21]:
# Models chosen based on BTZSC benchmark on the 28th March, 2026.
ZERO_SHOT_MODELS = [
# "Alibaba-NLP/gte-large-en-v1.5", excluded as remote code must be trusted and this is not recommended with sensitive data
# "Qwen/Qwen3-Reranker-0.6B", excluded as it is a retrieval/ranker model and not a drop-in for zero-shot classification
"intfloat/e5-large-v2",
"intfloat/e5-base-v2",
"cross-encoder/nli-deberta-v3-large"
]

In [5]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [6]:
# Load all data.
list_of_dfs = []
for folder in os.listdir(f'./{FILE_PATH}'):
    if os.path.isdir(f'./{FILE_PATH}/{folder}'):
        for file in os.listdir(f'./{FILE_PATH}/{folder}'):
            if file.endswith(".csv"):
                temp_file_path = f'./{FILE_PATH}/{folder}/{file}'
                temp_tag = file.replace('.csv', '')
                temp_df = pl.read_csv(temp_file_path)
                temp_df = (
                    temp_df
                    .with_row_index("index_num") # , offset=1) if you want to start index from 1
                    .with_columns(
                        (pl.lit(temp_tag) + "_" + pl.col("index_num").cast(pl.Utf8)).alias("doc_id")
                    )).select(['text', 'doc_id'])
                list_of_dfs.append(temp_df)

combined = pl.concat(list_of_dfs, how="vertical")
assert combined.select(pl.col("doc_id").n_unique()).item() == combined.height, "There should be no duplicates in the dataset."

In [7]:
# Remove invalid data.
temp_df = combined.with_columns(
    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
)
print(temp_df.group_by("tag").len())
print(f"Total Texts Before Empty String Removal: {len(temp_df)}")

temp_df = combined.with_columns(pl.col("text").str.strip_chars().alias("text")).filter(pl.col("text").is_not_null() & (pl.col("text") != ""))

temp_df = temp_df.with_columns(
    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
)
print(temp_df.group_by("tag").len())
print(f"Total Texts After Empty String Removal: {len(temp_df)}")

shape: (14, 2)
┌────────────────────────────────┬────────┐
│ tag                            ┆ len    │
│ ---                            ┆ ---    │
│ str                            ┆ u32    │
╞════════════════════════════════╪════════╡
│ banking77                      ┆ 13069  │
│ atis                           ┆ 4978   │
│ trumpTweets                    ┆ 56571  │
│ syntheticCareHomeNurseNotes    ┆ 5783   │
│ clinc150                       ┆ 23700  │
│ augmentedClinicalNotes         ┆ 30000  │
│ bbcNews                        ┆ 2225   │
│ huffPostNews                   ┆ 209527 │
│ simSUM                         ┆ 10000  │
│ dementiaAudio                  ┆ 549    │
│ 20NewsGroups                   ┆ 18846  │
│ yahoo                          ┆ 87362  │
│ medicalAbstracts               ┆ 14438  │
│ clinicalDialogueSummarizations ┆ 3603   │
└────────────────────────────────┴────────┘
Total Texts Before Empty String Removal: 480651
shape: (14, 2)
┌────────────────────────────────┬────────

In [8]:
# Randomly sample from dataframe.
temp_df = temp_df.with_columns(
    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
)
temp_df = temp_df.to_pandas()
temp_df = (temp_df.groupby("tag")).apply(lambda x: x.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE + batch_idx)) ; batch_idx += 1 # Increase batch_idx every time to ensure new random samples.
temp_df = pl.from_pandas(temp_df)


In [9]:
# Set up df for use.
df = pl.DataFrame({
    "doc_id": temp_df['doc_id'].to_list(),
    "text": temp_df['text'].to_list()
})

# Light preprocessing to strip extra whitespace.
df = df.with_columns(
    pl.col("text")
    .str.strip_chars()
    .str.replace_all(r"\s+", " ")
    .str.replace_all(r"^\s*-\s*", "") # Remove dashes at the beginning of texts.
    .str.replace_all(r"^\s*\d+\.\s*", "") # Remove numbers in 1., 2., 3. format at the beginning of the text. 
)

pybiber_pipeline = pb.PybiberPipeline(model="en_core_web_sm")
features, tokens = pybiber_pipeline.run(df, return_tokens=True)
features = features.with_columns(pl.col("doc_id").str.split("_").list.get(0).alias("category"))
# Full feature list can be found here: https://browndw.github.io/pybiber/feature-categories.html
print(f" -------- Features-------- ")
print(features)

# Statistical analysis and visualization
analyzer = pb.BiberAnalyzer(features, id_column='category')

# Multi-Dimensional Analysis - see https://browndw.github.io/pybiber/biber-analyzer.html#comparison-with-bibers-original-dimensions for factor mapping
# Explanation of the factor mapping to dimensions can be found here: https://www.uni-bamberg.de/fileadmin/eng-ling/fs/Chapter_21/23DimensionsofEnglish.html
'''
Factor 1: Involved vs. Informational Production (negative to positive)
Factor 2: Narrative vs. Non-narrative Concerns (negative to positive)
Factor 3: Explicit vs. Situation-dependent Reference (negative to positive)
Factor 4: Overt Expression of Persuasion (negative to positive)
Factor 5: Abstract vs. Non-abstract Information (negative to positive)
Factor 6: On-line Informational Elaboration (negative to positive)
'''

analyzer.mda_biber()
print(f" -------- MDA Summary -------- ")
print(analyzer.mda_summary)
print(f" -------- MDA Loadings -------- ")
print(analyzer.mda_loadings)
print(f" -------- MDA Dimension Scores -------- ")
print(analyzer.mda_dim_scores)
print(f" -------- MDA Group Means -------- ")
print(analyzer.mda_group_means)

[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length


 -------- Features-------- 
shape: (140, 69)
┌─────────┬─────────┬─────────┬─────────┬─────────┬───┬────────┬────────┬────────┬────────┬────────┐
│ doc_id  ┆ f_01_pa ┆ f_02_pe ┆ f_03_pr ┆ f_04_pl ┆ … ┆ f_64_p ┆ f_65_c ┆ f_66_n ┆ f_67_n ┆ catego │
│ ---     ┆ st_tens ┆ rfect_a ┆ esent_t ┆ ace_adv ┆   ┆ hrasal ┆ lausal ┆ eg_syn ┆ eg_ana ┆ ry     │
│ str     ┆ e       ┆ spect   ┆ ense    ┆ erbials ┆   ┆ _coord ┆ _coord ┆ thetic ┆ lytic  ┆ ---    │
│         ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆   ┆ inatio ┆ inatio ┆ ---    ┆ ---    ┆ str    │
│         ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆   ┆ n      ┆ n      ┆ f64    ┆ f64    ┆        │
│         ┆         ┆         ┆         ┆         ┆   ┆ ---    ┆ ---    ┆        ┆        ┆        │
│         ┆         ┆         ┆         ┆         ┆   ┆ f64    ┆ f64    ┆        ┆        ┆        │
╞═════════╪═════════╪═════════╪═════════╪═════════╪═══╪════════╪════════╪════════╪════════╪════════╡
│ 20NewsG ┆ 21.3675 ┆ 4.27350 ┆ 68.3760 ┆ 0.0 

In [10]:
os.makedirs(f"./{OUTPUT_DIR}/", exist_ok=True)

def flatten_for_csv(df):
    # Work on a copy to avoid modifying original.
    df_flat = df.clone()
    for c, dtype in zip(df_flat.columns, df_flat.dtypes):
        if dtype == pl.List:
            # Join list elements into string with commas.
            df_flat = df_flat.with_columns(
                pl.Series(df_flat[c].name, [",".join(map(str, x)) if x is not None else "" for x in df_flat[c]])
            )
        elif dtype == pl.Struct:
            # Convert struct to string representation.
            df_flat = df_flat.with_columns(
                pl.Series(df_flat[c].name, df_flat[c].cast(pl.Utf8))
            )
    return df_flat

# Get Z-Scores from Biber analysis.
biber_dimensions = (analyzer.mda_dim_scores).to_pandas()
print(analyzer.mda_dim_scores)
biber_dimensions = biber_dimensions.drop(columns=['factor_7']) # Remove factor 7 as it is not used or defined in Biber's original 6 dimensions.
# Get factor columns.
factor_cols = [c for c in biber_dimensions.columns if c.startswith("factor")]
biber_dimensions[factor_cols] = biber_dimensions[factor_cols].apply(zscore)
for c in factor_cols:
    biber_dimensions[f"{c}_label"] = biber_dimensions[c] > 0
biber_dimensions = pl.from_pandas(biber_dimensions)
print(biber_dimensions)

shape: (140, 9)
┌──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┐
│ doc_id   ┆ doc_cat  ┆ factor_1 ┆ factor_2 ┆ factor_3 ┆ factor_4 ┆ factor_5 ┆ factor_6 ┆ factor_7 │
│ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ str      ┆ str      ┆ f64      ┆ f64      ┆ f64      ┆ f64      ┆ f64      ┆ f64      ┆ f64      │
╞══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╡
│ 20NewsGr ┆ 20NewsGr ┆ 7.955458 ┆ -0.91233 ┆ -0.31370 ┆ -0.52738 ┆ 0.016249 ┆ 10.25006 ┆ 0.0      │
│ oups_113 ┆ oups     ┆          ┆ 2        ┆ 1        ┆          ┆          ┆ 8        ┆          │
│ 53       ┆          ┆          ┆          ┆          ┆          ┆          ┆          ┆          │
│ 20NewsGr ┆ 20NewsGr ┆ -0.66951 ┆ -0.75248 ┆ 10.70599 ┆ 1.370033 ┆ -1.16612 ┆ 1.538755 ┆ 0.0      │
│ oups_176 ┆ oups     ┆ 4        ┆ 6        ┆ 3        ┆          ┆ 1      

In [11]:
# Write CSV files.
flatten_for_csv(biber_dimensions).write_csv(f"./{OUTPUT_DIR}/mda_dim_scores{RANDOM_STATE + batch_idx - 1}.csv", float_precision=15)
flatten_for_csv(analyzer.mda_summary).write_csv(f"./{OUTPUT_DIR}/mda_summary{RANDOM_STATE + batch_idx - 1}.csv", float_precision=15)
flatten_for_csv(analyzer.mda_loadings).write_csv(f"./{OUTPUT_DIR}/mda_loadings{RANDOM_STATE + batch_idx - 1}.csv", float_precision=15)
flatten_for_csv(analyzer.mda_group_means).write_csv(f"./{OUTPUT_DIR}/mda_group_means{RANDOM_STATE + batch_idx - 1}.csv", float_precision=15)

# Write JSON files.
analyzer.mda_summary.write_json(f"./{OUTPUT_DIR}/mda_summary{RANDOM_STATE + batch_idx - 1}.json")
analyzer.mda_loadings.write_json(f"./{OUTPUT_DIR}/mda_loadings{RANDOM_STATE + batch_idx - 1}.json")
analyzer.mda_dim_scores.write_json(f"./{OUTPUT_DIR}/mda_dim_scores{RANDOM_STATE + batch_idx - 1}.json")
analyzer.mda_group_means.write_json(f"./{OUTPUT_DIR}/mda_group_means{RANDOM_STATE + batch_idx - 1}.json")

In [12]:
# Exact mapping taken from https://www.uni-bamberg.de/fileadmin/eng-ling/fs/Chapter_21/23DimensionsofEnglish.html which has extracted the same from Biber and Conrad's Variation in English (https://doi.org/10.4324/9781315840888)
BIBER_LABEL_MAP = {
    "factor_1": ["informational", "involved"],
    "factor_2": ["non_narrative", "narrative"],
    "factor_3": ["situation_dependent", "explicit and elaborated"],
    "factor_4": ["non_persuasive and non_overtly_argumentative", "persuasive and overtly_argumentative"],
    "factor_5": ["non_abstract and concrete", "abstract"],
    "factor_6": ["compressed", "elaborated"],
}

# Using different prompt templates increases robustness. 
TEMPLATES = [
    "This text is {}.",
    "This text is written in a {} style.",
    "The writing style of this text is {}.",
    "This text can be described as {}.",
    "Based on Douglas Biber's dimensions, this text is written in a {} style."
]

In [13]:
temp_df = temp_df.to_pandas()
texts = temp_df['text'].values.tolist()[:10]

In [19]:
def create_zero_shot_pipeline(model_name, max_length=512, device=DEVICE):
    """
    Creates a zero-shot classification pipeline with automatic truncation.
    """
    # Load tokenizer with truncation
    tokenizer = AutoTokenizer.from_pretrained(model_name, truncation=True, trust_remote_code=True)
    
    # Load model
    model = AutoModelForSequenceClassification.from_pretrained(model_name, trust_remote_code=True)
    
    # Wrap into pipeline with tokenizer kwargs
    classifier = pipeline(
        "zero-shot-classification",
        model=model,
        tokenizer=tokenizer,
        device=-1,
        tokenizer_kwargs={"truncation": True, "max_length": max_length},
        trust_remote_code=True  # needed for some custom models like Qwen or Alibaba
    )
    
    return classifier

In [24]:
pipelines = {}
for model_name in ZERO_SHOT_MODELS:
    print(f"Loading model: {model_name}")
    classifier = pipeline(
        "zero-shot-classification",
        model=model_name,
        device=-1
    )
    outputs = classifier(
        texts,
        candidate_labels=BIBER_LABEL_MAP['factor_1'],
        hypothesis_template=TEMPLATES[0],
        batch_size=8
    )

    # Free memory.
    del classifier
    gc.collect()

    break

Loading model: intfloat/e5-large-v2


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: intfloat/e5-large-v2
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Failed to determine 'entailment' label id from the label2id mapping in the model config. Setting to -1. Define a descriptive label2id mapping in the model config to ensure correct outputs.


In [ ]:
# Using the pipeline on some text
texts = ["This text describes a technical report about AI research."]
example_model = ZERO_SHOT_MODELS[0]
outputs = pipelines[example_model](
    texts[0],
    candidate_labels=["involved", "informational"],
    hypothesis_template="This example is {}."
)
print(outputs)